In [1]:
SKIP_LLM_CHUNKING = True

## Ingestion


In [2]:
import io
import zipfile
import requests
import frontmatter

def read_repo_data(repo_owner, repo_name):
    """
    Download and parse all markdown files from a GitHub repository.
    
    Args:
        repo_owner: GitHub username or organization
        repo_name: Repository name
    
    Returns:
        List of dictionaries containing file content and metadata
    """
    prefix = 'https://codeload.github.com' 
    url = f'{prefix}/{repo_owner}/{repo_name}/zip/refs/heads/main'
    resp = requests.get(url)
    
    if resp.status_code != 200:
        raise Exception(f"Failed to download repository: {resp.status_code}")

    repository_data = []
    zf = zipfile.ZipFile(io.BytesIO(resp.content))
    
    for file_info in zf.infolist():
        filename = file_info.filename
        filename_lower = filename.lower()

        if not (filename_lower.endswith('.md') 
            or filename_lower.endswith('.mdx')):
            continue
    
        try:
            with zf.open(file_info) as f_in:
                content = f_in.read().decode('utf-8', errors='ignore')
                post = frontmatter.loads(content)
                data = post.to_dict()
                data['filename'] = filename
                repository_data.append(data)
        except Exception as e:
            print(f"Error processing {filename}: {e}")
            continue
    
    zf.close()
    return repository_data

In [3]:
dtc_faq = read_repo_data('DataTalksClub', 'faq')
evidently_docs = read_repo_data('evidentlyai', 'docs')

print(f"FAQ documents: {len(dtc_faq)}")
print(f"Evidently documents: {len(evidently_docs)}")

FAQ documents: 1285
Evidently documents: 95


## Chunking and Preprocessing

### Intelligent Chunking with LLM

In [4]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()

# now you can access them
openai_key = os.getenv("OPENAI_API_KEY")
print("Has key?", bool(openai_key))

if not SKIP_LLM_CHUNKING:
    openai_client = OpenAI(openai_key)

    def llm(prompt, model='gpt-4o-mini'):
        messages = [
            {"role": "user", "content": prompt}
        ]

        response = openai_client.responses.create(
            model='gpt-4o-mini',
            input=messages
        )

        return response.output_text


Has key? True


In [5]:
prompt_template = """
Split the provided document into logical sections
that make sense for a Q&A system.

Each section should be self-contained and cover
a specific topic or concept.

<DOCUMENT>
{document}
</DOCUMENT>

Use this format:

## Section Name

Section content with all relevant details

---

## Another Section Name

Another section content

---
""".strip()


In [6]:
if not SKIP_LLM_CHUNKING:
    def intelligent_chunking(text):
        prompt = prompt_template.format(document=text)
        response = llm(prompt)
        sections = response.split('---')
        sections = [s.strip() for s in sections if s.strip()]
        return sections

In [7]:
from tqdm.auto import tqdm

if not SKIP_LLM_CHUNKING:
    evidently_chunks = []

    for doc in tqdm(evidently_docs):
        doc_copy = doc.copy()
        doc_content = doc_copy.pop('content')

        sections = intelligent_chunking(doc_content)
        for section in sections:
            section_doc = doc_copy.copy()
            section_doc['section'] = section
            evidently_chunks.append(section_doc)

### Simple Chunking

In [8]:
def sliding_window(seq, size, step):
  if size <= 0 or step <= 0:
    raise ValueError("size and step must be positive")

  n = len(seq)
  result = []

  for i in range(0, n, step):
    chunk = seq[i:i+size]
    result.append({'start': i, 'chunk': chunk})

    if i + size >= n:
      break
  
  return result

In [9]:
from tqdm.auto import tqdm

evidently_chunks = []
for doc in evidently_docs:
  doc_copy = doc.copy()
  doc_content = doc_copy.pop('content')
  chunks = sliding_window(doc_content, 2000, 1000)

  for chunk in chunks:
    chunk.update(doc_copy)

  evidently_chunks.extend(chunks)
  
print(f"Evidently chunks: {len(evidently_chunks)}")

Evidently chunks: 576


## Search

### Text search

In [10]:
from minsearch import Index

evidently_index = Index(
    text_fields=["chunk", "title", "description", "filename"],
    keyword_fields=[]
)

evidently_index.fit(evidently_chunks)

In [11]:
query = 'What should be in a test dataset for AI evaluation?'
results = evidently_index.search(query)

print(f"Top result: {results[0]['chunk'][:500]}...")

Top result: Retrieval-Augmented Generation (RAG) systems rely on retrieving answers from a knowledge base before generating responses. To evaluate them effectively, you need a test dataset that reflects what the system *should* know.

Instead of manually creating test cases, you can generate them directly from your knowledge source, ensuring accurate and relevant ground truth data.

## Create a RAG test dataset

You can generate ground truth RAG dataset from your data source.

### 1. Create a Project

In th...


### Vector search

In [12]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer('multi-qa-distilbert-cos-v1')

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

In [13]:
from minsearch import VectorSearch
from tqdm.auto import tqdm
import numpy as np

evidently_embeddings = []

for d in tqdm(evidently_chunks):
  v = embedding_model.encode(d['chunk'])
  evidently_embeddings.append(v)

evidently_embeddings = np.array(evidently_embeddings)

evidently_vindex = VectorSearch()
evidently_vindex.fit(evidently_embeddings, evidently_chunks)

  0%|          | 0/576 [00:00<?, ?it/s]

In [14]:
print(evidently_embeddings.shape)
print(len(evidently_chunks))

(576, 768)
576


In [15]:
query = 'What should be in a test dataset for AI evaluation?'
q = embedding_model.encode(query)
results = evidently_vindex.search(q)

print(f"Top result: {results[0]['chunk'][:500]}...")

Top result: When working on an AI system, you need test data to run automated evaluations for quality and safety. A test dataset is a structured set of test cases. It can contain:

* Just the inputs, or
* Both inputs and expected outputs (ground truth).

You can use this test dataset to:

* Run **experiments** and track if changes improve or degrade system performance.
* Run **regression testing** to ensure updates don’t break what was already working.
* **Stress-test** your system with complex or adversari...


### Hybrid search

In [16]:
def pure_text_search(query, num_results):
  return evidently_index.search(query, num_results=num_results)

def vector_search(query, num_results):
  q = embedding_model.encode(query)
  return evidently_vindex.search(q, num_results=num_results)

def hybrid_search(query, num_results):
  text_results = pure_text_search(query, num_results)
  vector_results = vector_search(query, num_results)

  # Combine and deduplicate results
  seen_ids = set()
  combined_results = []

  for result in text_results + vector_results:
    if result['filename'] not in seen_ids:
      seen_ids.add(result['filename'])
      combined_results.append(result)

  return combined_results

## Agents and Tools

In [17]:
system_prompt = """
You are a helpful assistant about Evidently - an open-source Python library to evaluate, test, and monitor ML and LLM systems, from experiments to production.

Use the search tool to find relevant information before answering questions.

If you can find specific information through search, use it to provide accurate
answers.

If the search doesn't return relevant results, let the user know and provide
general guidance
"""

In [18]:
from typing import List, Any
def text_search(query: str) -> List[Any]:
  """
  Perform a text-based search on the Evidently index.
  Args:
    query (str): The search query string.
  Returns:
    List[Any]: A list of up to 5 search results returned by the Evidently index.
  """
  return hybrid_search(query, num_results=5)

In [19]:
from pydantic_ai import Agent
from pydantic_ai import Agent

agent = Agent(
  name="evidently_agent",
  instructions=system_prompt,
  tools=[text_search],
  model='gpt-4o-mini'
)

c:\Projects\Courses\aihero-jm\.venv\Lib\site-packages\pydantic_ai\models\__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-4o-mini', use 'openai:gpt-4o-mini'.
  provider_name, model_name = parse_model_id(model)


In [20]:
%load_ext dotenv
%dotenv .env

question = "What should be in a test dataset for AI evaluation?"
result = await agent.run(user_prompt=question)

print(result)

AgentRunResult(output="When creating a test dataset for the evaluation of AI systems, especially in the context of machine learning or large language models (LLM), you should consider the following key components:\n\n1. **Diversity of Input Cases**: Ensure that the dataset includes a variety of input cases, covering different scenarios, edge cases, and potential user interactions. This helps evaluate how well the AI system can generalize from the training data to new situations.\n\n2. **Ground Truth Labels**: The test dataset must include accurate ground truth labels or expected outputs for the corresponding inputs. This is crucial for assessing the performance of the AI system against these benchmarks.\n\n3. **Synthetic Data Generation**: In cases where real data is scant or biased, synthetic data can be generated to fill in gaps. This allows for testing specific features or handling of adversarial inputs effectively. However, it should not replace real data entirely.\n\n4. **Context 

## Evaluation

### Logging

In [21]:
from pydantic_ai.messages import ModelMessagesTypeAdapter

def log_entry(agent, messages, source="user"):
  tools = []
  
  for ts in agent.toolsets:
    tools.extend(ts.tools.keys())

  dict_messages = ModelMessagesTypeAdapter.dump_python(messages)
  
  return {
    "agent_name": agent.name,
    "system_prompt": agent._instructions,
    "provider": agent.model.system,
    "model": agent.model.model_name,
    "tools": tools,
    "messages": dict_messages,
    "source": source
  }

In [22]:
import json
import secrets
from pathlib import Path
from datetime import datetime

LOG_DIR = Path('logs')
LOG_DIR.mkdir(exist_ok=True)

def serializer(obj):
  if isinstance(obj, datetime):
    return obj.isoformat()
  raise TypeError(f"Type {type(obj)} not serializable")

def log_interaction_to_file(agent, messages, source='user'):
  entry = log_entry(agent, messages, source)

  ts = entry['messages'][-1]['timestamp']

  if not isinstance(ts, datetime):
    ts = datetime.fromisoformat(str(ts).replace("Z", "+00:00"))

  ts_str = ts.strftime("%Y%m%d_%H%M%S")
  rand_hex = secrets.token_hex(3)

  filename = f"{agent.name}_{ts_str}_{rand_hex}.json"
  filepath = LOG_DIR / filename
  
  with filepath.open("w", encoding="utf-8") as f_out:
    json.dump(entry, f_out, indent=2, default=serializer)

  return filepath

In [23]:
question = "What should be in a test dataset for AI evaluation?" # input()

result = await agent.run(user_prompt=question)
print(result.output)

log_interaction_to_file(agent, result.new_messages())

Creating a test dataset for AI evaluation involves several important considerations to ensure that the dataset accurately reflects the intended use cases and scenarios relevant to the AI system being evaluated. Here are some key elements to include in a test dataset for AI evaluation:

1. **Diversity of Scenarios**: The dataset should encompass a variety of scenarios, including typical cases, edge cases, and potential failure modes. This helps in evaluating the robustness of the AI system.

2. **Ground Truth Data**: Ensure that the dataset contains accurate ground truth labels or outputs against which the AI's predictions can be evaluated. This is crucial for assessing performance metrics.

3. **Relevance to Use Cases**: The data should be representative of actual usage conditions to evaluate how the AI performs in real-world situations.

4. **Adversarial Examples**: Including adversarial inputs can help test the AI system's resilience to input that is specifically designed to confuse 

WindowsPath('logs/evidently_agent_20260330_115158_a92c23.json')

### Adding References

In [24]:
system_prompt = """
You are a helpful assistant about Evidently - an open-source Python library to evaluate, test, and monitor ML and LLM systems, from experiments to production.

Use the search tool to find relevant information before answering questions.

If you can find specific information through search, use it to provide accurate answers.

Always include references by citing the filename of the source material you used.

When citing the reference, replace "evidently-main" by the full path to the GitHub repository: "https://github.com/evidentlyai/evidently/blob/main/"
Format: [LINK TITLE](FULL_GITHUB_LINK)

If the search doesn't return relevant results, let the user know and provide general guidance.
""".strip()

# Create another version of agent, let's call it faq_agent_v2
agent = Agent(
  name="evidently_agent_v2",
  instructions=system_prompt,
  tools=[text_search],
  model='gpt-4o-mini'
)

c:\Projects\Courses\aihero-jm\.venv\Lib\site-packages\pydantic_ai\models\__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-4o-mini', use 'openai:gpt-4o-mini'.
  provider_name, model_name = parse_model_id(model)


In [25]:
question = "What should be in a test dataset for AI evaluation?" # input()

result = await agent.run(user_prompt=question)
print(result.output)

log_interaction_to_file(agent, result.new_messages())

A test dataset for AI evaluation should include various essential components to ensure comprehensive and effective assessment of the AI system's performance. Here are the key elements to include:

1. **Diverse Inputs**: The dataset should cover a wide range of scenarios and inputs that the model might encounter in real-world applications. This includes normal cases, edge cases, and adversarial examples to test the model's robustness.

2. **Ground Truth Labels**: Each input should be paired with accurate and reliable outputs or responses (ground truth). This is crucial for evaluating the performance of the AI system against an expected standard.

3. **Synthetic Data Generation**: For scenarios where real data is scarce or where specific testing of functionalities is required, synthetic data can be useful. This enables the generation of structured test cases that add variety and help fill gaps in the testing dataset. 

4. **Evaluation Metrics**: The dataset should be structured to allow 

WindowsPath('logs/evidently_agent_v2_20260330_115211_50744f.json')

### LLM as judge

In [26]:
evaluation_prompt = """
Use this checklist to evaluate the quality of an AI agent's answer (<ANSWER>) to a user question (<QUESTION>).

We also include the entire log (<LOG>) for analysis.

For each item, check if the condition is met.
Checklist:
- instructions_follow: The agent followed the user's instructions (in <INSTRUCTIONS>)
- instructions_avoid: The agent avoided doing things it was told not to do
- answer_relevant: The response directly addresses the user's question
- answer_clear: The answer is clear and correct
- answer_citations: The response includes proper citations or sources when required
- completeness: The response is complete and covers all key aspects of the request
- tool_call_search: Is the search tool invoked?

Output true/false for each check and provide a short explanation for your judgment.
""".strip()

In [27]:
from pydantic import BaseModel

class EvaluationCheck(BaseModel):
  check_name: str
  justification: str
  check_pass: bool
  
class EvaluationChecklist(BaseModel):
  checklist: list[EvaluationCheck]
  summary: str
  
eval_agent = Agent(
  name='eval_agent',
  model='gpt-5-nano',
  instructions=evaluation_prompt,
  output_type=EvaluationChecklist
)

c:\Projects\Courses\aihero-jm\.venv\Lib\site-packages\pydantic_ai\models\__init__.py:1280: DeprecationWarning: Specifying a model name without a provider prefix is deprecated. Instead of 'gpt-5-nano', use 'openai:gpt-5-nano'.
  provider_name, model_name = parse_model_id(model)


In [28]:
user_prompt_format = """
<INSTRUCTIONS>{instructions}</INSTRUCTIONS>
<QUESTION>{question}</QUESTION>
<ANSWER>{answer}</ANSWER>
<LOG>{log}</LOG>
""".strip()

In [29]:
def load_log_file(log_file):
  with open(log_file, 'r') as f_in:
    log_data = json.load(f_in)
    log_data['log_file'] = log_file
    return log_data

In [30]:
log_record = load_log_file('./logs/evidently_agent_20260329_210039_b82639.json')

instructions = log_record['system_prompt']
question = log_record['messages'][0]['parts'][0]['content']
answer = log_record['messages'][-1]['parts'][0]['content']
log = json.dumps(log_record['messages'])

user_prompt = user_prompt_format.format(
  instructions=instructions,
  question=question,
  answer=answer,
  log=log
)

In [31]:
result = await eval_agent.run(user_prompt, output_type=EvaluationChecklist)

checklist = result.output
print(checklist.summary)

for check in checklist.checklist:
  print(check)

Tool call initiated to fetch information about test datasets for AI evaluation; prepared to synthesize results into a concise answer.
check_name='instructions_follow' justification='We will search first per given instruction; we will respond with information about test dataset after tool results.' check_pass=True
check_name='instructions_avoid' justification='No disallowed content.' check_pass=True
check_name='answer_relevant' justification='The user asked for what should be in a test dataset; the answer will provide a structured list.' check_pass=True
check_name='answer_clear' justification='Will provide clear bullet points with rationale.' check_pass=True
check_name='answer_citations' justification='No explicit citations included. Only general guidance; no required citations.' check_pass=True
check_name='completeness' justification='Covers key aspects of test datasets for AI evaluation.' check_pass=True
check_name='tool_call_search' justification='A tool call to text_search will be p